## Abgleich scenario_name
Relevant, weil peq-higrube modell-erstellung mit gleichen typen/scenario_names arbeiten muss

Bestandteile je Position (separatd by "|" vergleichen).

referenz_datei ist original-format, ziel_datei ist zu verändern

ziel-datei wird dann neu als scenarios_harmonisiert.parquet abgespeichert

In [4]:
from pathlib import Path
from difflib import get_close_matches
import pandas as pd
from IPython.display import display

referenz_datei = Path(r"C:\Users\ma0026.NB-MA0026\Documents\peexcel\data\building_typologies\building_sector_v2\scenarios.parquet")
ziel_datei = Path(r"C:\Users\ma0026.NB-MA0026\Documents\peexcel\data\building_typologies\building_sector_v3\scenarios.parquet")
sep = "|"

ref = pd.read_parquet(referenz_datei, columns=["scenario_name"])["scenario_name"]
zie = pd.read_parquet(ziel_datei, columns=["scenario_name"])["scenario_name"]

def zerlegen(serie):
    teile = serie.str.split(sep, regex=False)
    return teile

ref_t, zie_t = zerlegen(ref), zerlegen(zie)
n_ref, n_zie = ref_t.map(len), zie_t.map(len)
print("Komponenten Referenz:", n_ref.value_counts().to_dict())
print("Komponenten Ziel:    ", n_zie.value_counts().to_dict())
if n_ref.nunique() != 1 or n_zie.nunique() != 1 or n_ref.iloc[0] != n_zie.iloc[0]:
    raise ValueError("Unterschiedliche Komponentenanzahl; Namen zuerst strukturell angleichen.")

n = n_ref.iloc[0]
ref_tokens = [sorted({t[i] for t in ref_t}) for i in range(n)]
zie_tokens = [sorted({t[i] for t in zie_t}) for i in range(n)]

zeilen = []
for i in range(n):
    norm = {" ".join(r.split()).casefold(): r for r in ref_tokens[i]}
    for tok in zie_tokens[i]:
        if tok in ref_tokens[i]:
            continue
        vorschlag = norm.get(" ".join(tok.split()).casefold())
        if vorschlag is None:
            aehnlich = get_close_matches(tok.casefold(), list(norm), n=3, cutoff=0.4)
            vorschlag = [norm[a] for a in aehnlich]
        zeilen.append({"Position": i, "Ziel": tok, "Vorschlag": vorschlag,
                       "Referenz_Optionen": ref_tokens[i] if len(ref_tokens[i]) <= 12 else "..."})

bericht = pd.DataFrame(zeilen)
display(bericht if len(bericht) else "Alle Bestandteile stimmen überein.")

Komponenten Referenz: {7: 43200}
Komponenten Ziel:     {7: 43200}


,Position,Ziel,Vorschlag,Referenz_Optionen
0,1,Neubau10cm,"[Neu_10cm, Neu_30cm, Neu_20cm]","[1919-44, 1945-59, 1960-79, 1980-89, 1990-99, ..."
1,1,Neubau20cm,"[Neu_20cm, Neu_30cm, Neu_10cm]","[1919-44, 1945-59, 1960-79, 1980-89, 1990-99, ..."
2,1,Neubau30cm,"[Neu_30cm, Neu_20cm, Neu_10cm]","[1919-44, 1945-59, 1960-79, 1980-89, 1990-99, ..."
3,2,Biomasse,"[Bio, WP_Wasser]","[Bio, Elektro, Erdgas, FW_Wien, FW_erneu, Oel,..."
4,2,FW_Innsbruck,"[FW_Wien, FW_erneu]","[Bio, Elektro, Erdgas, FW_Wien, FW_erneu, Oel,..."
5,2,FW_erneuerbar,"[FW_erneu, FW_Wien]","[Bio, Elektro, Erdgas, FW_Wien, FW_erneu, Oel,..."
6,2,WP_Freecooling,"[WP_Sole, WP_Sole_fc]","[Bio, Elektro, Erdgas, FW_Wien, FW_erneu, Oel,..."
7,2,WP_Grundwasser,"[WP_Wasser, WP_Sole, WP_Sole_fc]","[Bio, Elektro, Erdgas, FW_Wien, FW_erneu, Oel,..."
8,5,Innsbruck,Innsbruck,"[Innsbruck , Wien ]"
9,5,Wien,Wien,"[Innsbruck , Wien ]"


## mapping festlegen

Die Zuordnung erfolgt je Position. Damit wird verhindert, dass ein Kürzel in einer anderen Komponente versehentlich mitersetzt wird.

In [9]:
# Position -> {alter Bestandteil: Referenz-Bestandteil}
mapping = {
    1: {"Neubau10cm": "Neu_10cm", "Neubau20cm": "Neu_20cm", "Neubau30cm": "Neu_30cm"},
    2: {"Biomasse": "Bio", "FW_Innsbruck": "FW_Wien", "FW_erneuerbar": "FW_erneu", "WP_Freecooling":"WP_Sole_fc", "WP_Grundwasser": "WP_Wasser"},
    5: {"Innsbruck": "Innsbruck ", "Wien": "Wien "}
}
vorschau = False
ausgabe = ziel_datei.with_name(ziel_datei.stem + "_harmonisiert.parquet")

for pos, m in mapping.items():
    fehlend = set(m.values()) - set(ref_tokens[pos])
    if fehlend:
        raise ValueError(f"Position {pos}: Zielwerte nicht in Referenz: {fehlend}")

neu = zie_t.map(lambda t: sep.join(mapping.get(i, {}).get(x, x) for i, x in enumerate(t)))

# Neue Kollisionen: verschiedene alte Namen -> gleicher neuer Name
kombi = pd.DataFrame({"alt": zie, "neu": neu}).drop_duplicates()
kol = kombi[kombi["neu"].duplicated(keep=False)]
if len(kol):
    display(kol.sort_values("neu"))
    raise ValueError("Mapping erzeugt Namenskollisionen.")

# Abgleich mit Referenz
ref_set = set(ref)
passt = neu.isin(ref_set)
print(f"Exakt in Referenz: {passt.sum():,}/{len(neu):,} ({passt.mean():.1%})")
print(f"Referenznamen ohne Entsprechung: {len(ref_set - set(neu)):,}")
display(pd.Series(neu[~passt].unique(), name="ohne_Treffer").head(20))

if vorschau:
    print("Nur Vorschau. Zum Speichern: vorschau = False.")
else:
    if ausgabe.exists():
        raise FileExistsError(ausgabe)
    df = pd.read_parquet(ziel_datei)
    df["scenario_name"] = neu
    df.to_parquet(ausgabe, index=False)
    print("Gespeichert:", ausgabe)

Exakt in Referenz: 43,200/43,200 (100.0%)
Referenznamen ohne Entsprechung: 0


Series([], Name: ohne_Treffer, dtype: str)

Gespeichert: C:\Users\ma0026.NB-MA0026\Documents\peexcel\data\building_typologies\building_sector_v3\scenarios_harmonisiert.parquet
